In [ ]:
import pandas as pd
import numpy as np

import torch
from botorch.models import SingleTaskGP
from botorch.acquisition import qUpperConfidenceBound
from botorch.optim import optimize_acqf_discrete_local_search

from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

In [9]:
def data_convert(df:pd.DataFrame):
    target_cols =['Nluc template', 'Mg(OAc)2', 'NTPs', 'cap2','HEPES-KOH','pyrophosphatase', 'capped mRNA µg/µL']
    df = df[df["Capping efficiency"] != -1]
    df=df[target_cols]
    col_convert = {'Nluc template':'x0','Mg(OAc)2':'x1','NTPs':'x2','cap2':'x3','HEPES-KOH':'x4','pyrophosphatase':'x5','capped mRNA µg/µL':'y'}
    df = df.rename(columns=col_convert)
    return df

def run_bayes(data:pd.DataFrame,steps:list,maxs:list,mins:list,beta:float=2.0):
    # 実験における変数の数を定義
    num_x = len(data.columns)-1
    # 各変数の刻み幅(step),最小値(min),最大値(max),標準化後の刻み幅(norm_step),標準化後の定義域(norm_range)を定義
    norm_x_steps = []
    norm_x_ranges = []
    for i in range(num_x):
        tmp_step = steps[i]/(maxs[i]-mins[i])
        tmp_range = torch.Tensor(np.arange(0,1+tmp_step,tmp_step))
        norm_x_steps.append(tmp_step)
        norm_x_ranges.append(tmp_range)
    print(f"norm_x_steps:{norm_x_steps}")

    mmscaler = MinMaxScaler()
    norm_x = mmscaler.fit_transform(data.iloc[:,:-1])
    norm_x_df = pd.DataFrame(norm_x)
    y = data.iloc[:,-1].values
    norm_x_df.insert(len(data.columns)-1,'y',-y)
    norm_x_df.columns = data.columns
    norm_x_df
    # norm_x_df.to_csv('./data/round1_normed.csv')
    # norm_data = pd.read_csv("./data/round1_normed.csv", encoding="shift_jis",index_col=0)
    num_x = norm_x_df.shape[1]-1

    # 標準化済みのcsvデータ（既知の実験データ）を読み込む
    # norm_data = pd.read_csv("./data/round1_normed.csv", encoding="shift_jis",index_col=0)
    norm_data_x = norm_x_df.iloc[:-2,0:num_x]        #全行のxを読み込む
    norm_data_y = norm_x_df.iloc[:-2,num_x:num_x+1]  #全行のyを読み込む

    X = torch.Tensor(norm_data_x.values)
    Y = -torch.Tensor(norm_data_y.values)

    gp = SingleTaskGP(X, Y)
    qUCB = qUpperConfidenceBound(gp, beta=beta)
    candidate, acq_value = optimize_acqf_discrete_local_search(
        qUCB, discrete_choices=norm_x_ranges,q=20, num_restarts=20, raw_samples=len(data)-2,
    )
    inv_cand = mmscaler.inverse_transform(candidate)
    inv_cand = np.round(inv_cand,3)

    return inv_cand

In [10]:
raw_data_round1 = pd.read_csv(
    "../data/new_conditions/round1/round1_wet_result.csv",
    index_col=0,
).T
raw_data_init = pd.read_csv(
    "../data/new_conditions/round_init_exp2.csv",
    index_col=0,
).T
# raw_data  = raw_data.apply(pd.to_numeric, errors='coerce') # convert dtype from object to float
round1 = data_convert(raw_data_round1)
init = data_convert(raw_data_init)

In [11]:
data = pd.concat([round1,init],axis=0)

In [12]:
#mins = data.describe().loc['min'].values
#maxs = data.describe().loc['max'].values
#['Nluc template', 'Mg(OAc)2', 'NTPs', 'cap2', 'HEPES-KOH', 'pyrophosphatase', 'capped mRNA µg/µL']

maxs = [100,120,25,1.5,80,10,0]
mins = [10,8,5,0.1,10,1,0]
#maxs = [90,100,6,3,2]
data.loc['min']=mins
data.loc['max']=maxs
# ['Nluc template', 'Mg(OAc)2', 'NTPs', 'cap2', 'HEPES-KOH', 'pyrophosphatase', 'capped mRNA µg/µL']
steps = [0.5,0.5,0.5,0.1,0.5,0.5,0.01] # more detailed step
#steps = [5.0,1.0,1.0,0.5,0.01]

cands = run_bayes(data,steps,maxs,mins)
cands

norm_x_steps:[0.005555555555555556, 0.004464285714285714, 0.025, 0.07142857142857144, 0.007142857142857143, 0.05555555555555555]


array([[ 79.5,  60.5,  10.5,   0.8,  49. ,   5.5],
       [ 69.5,  55. ,  10. ,   0.9,  46. ,   7.5],
       [ 90. ,  69. ,  11.5,   0.9,  59. ,   5.5],
       [ 79. ,  69. ,  10. ,   0.8,  42.5,   3.5],
       [ 94. ,  57. ,  10.5,   0.6,  50. ,   6. ],
       [ 70. ,  49.5,   7.5,   0.8,  60. ,   5.5],
       [ 65.5,  61. ,  10.5,   0.8,  34.5,   5.5],
       [ 90.5,  60. ,   7. ,   0.9,  42.5,   7. ],
       [ 79.5,  61.5,  11.5,   0.6,  62.5,   5.5],
       [ 81. ,  76. ,  10. ,   1.1,  47. ,   5.5],
       [ 68. ,  45. ,  13. ,   0.9,  52.5,   8. ],
       [ 87.5,  65.5,  11. ,   0.8,  61.5,   3.5],
       [ 77. ,  71.5,  15. ,   0.9,  64. ,   5.5],
       [ 67. ,  33. ,   8. ,   0.7,  32. ,   7.5],
       [ 88. ,  52.5,  10.5,   0.6,  39.5,   6.5],
       [ 90. ,  53.5,   8.5,   0.7,  40. ,   5.5],
       [ 98.5,  79.5,  12.5,   0.6,  40.5,   4.5],
       [ 93. ,  68.5,  10.5,   0.6,  23.5,   6. ],
       [100. ,  72.5,  13.5,   0.7,  37.5,   1.5],
       [ 80.5,  46. ,   7.5,   

In [23]:
cands = run_bayes(data, steps, maxs, mins,beta=1.0)
cands

array([[80. , 57.5, 10.5,  0.8, 48.5,  6. ],
       [87. , 63. , 10.5,  0.8, 50.5,  4.5],
       [70.5, 54.5,  9.5,  0.9, 44.5,  7. ],
       [89. , 59.5, 11. ,  0.8, 57. ,  6.5],
       [77. , 52.5, 10. ,  0.6, 42. ,  6. ],
       [74.5, 52.5,  8.5,  0.9, 57. ,  6. ],
       [89.5, 70.5, 10. ,  0.9, 44.5,  6. ],
       [77. , 64.5, 11.5,  0.8, 43.5,  5. ],
       [80. , 57.5, 11.5,  0.8, 45.5,  7. ],
       [81. , 47. ,  8.5,  0.8, 36. ,  6.5],
       [65. , 68. , 10. ,  0.8, 35.5,  7. ],
       [94. , 55.5,  7.5,  0.7, 49. ,  5.5],
       [75. , 45. , 10. ,  1. , 38. ,  9. ],
       [81. , 65. , 10.5,  1. , 43. ,  7.5],
       [91. , 48.5, 10.5,  0.9, 42. ,  6. ],
       [78.5, 68.5, 11. ,  0.9, 49. ,  6. ],
       [81.5, 61. , 11. ,  0.7, 54.5,  3.5],
       [83.5, 64.5, 12.5,  0.7, 48. ,  6.5],
       [86. , 70.5,  9. ,  0.7, 43.5,  5. ],
       [72.5, 57.5, 11.5,  0.8, 53.5,  7.5]])

In [21]:
index = [f'cand{i}' for i in range(1,len(cands)+1)]
cand_df = pd.DataFrame(cands,index=index,columns=data.columns[:-1])
col_convert = {'Nluc template':'x0','Mg(OAc)2':'x1','NTPs':'x2','cap2':'x3','HEPES-KOH':'x4','pyrophosphatase':'x5','capped mRNA µg/µL':'y'}
col_conv_rev = {v:k for k,v in col_convert.items()}
cand_df=cand_df.rename(columns=col_conv_rev)
cand_df.to_csv('../outputs/round2_cand_qUCB_beta2.csv')

In [22]:
cand_df

,Nluc template,Mg(OAc)2,NTPs,cap2,HEPES-KOH,pyrophosphatase
cand1,80.5,59.0,10.0,0.8,49.0,6.0
cand2,67.5,49.5,10.0,0.8,42.5,7.5
cand3,92.0,66.5,11.0,0.7,51.0,4.5
cand4,82.5,64.0,10.0,1.0,60.0,7.0
cand5,69.0,66.0,8.0,0.9,38.5,5.5
cand6,97.5,52.5,9.5,0.7,47.0,7.0
cand7,76.5,52.0,7.5,0.9,60.0,4.5
cand8,78.0,55.0,13.0,0.6,57.0,6.5
cand9,77.0,58.0,8.5,0.5,45.0,6.0
cand10,99.5,81.5,10.5,1.0,49.0,5.0
